In [1]:
%pip install langchain-community==0.4.1
%pip install langchain-text-splitters==1.0.0
%pip install langchain-openai==1.1.0
%pip install langsmith==0.4.49
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install langchain==1.1.0
%pip install beautifulsoup4==4.14.2
%pip install python-dotenv==1.2.1

Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain_text_splitters-1.0.0-py3-none-any.whl (33 kB)
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.5 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 1.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use

In [3]:
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
from langchain_community.document_loaders import WebBaseLoader
import bs4
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

In [4]:
# OpenAI Setup
_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']

In [5]:
#### INDEXING ####

In [6]:
# Load Documents
loader = WebBaseLoader(
    web_paths=("https://kbourne.github.io/chapter1.html",), 
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [7]:
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)
splits = text_splitter.split_documents(docs)

In [8]:
# Embed
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

In [9]:
#### RETRIEVAL and GENERATION ####

In [10]:
# Prompt
client = Client()
prompt = client.pull_prompt("jclemens24/rag-prompt")

/Users/prashantsharma/GitHub/Unlocking-Data-with-Generative-AI-and-RAG-Second-Edition/.venv/lib/python3.10/site-packages/langsmith/client.py:7775: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


In [11]:
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
# LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

In [13]:
# Chain it all together with LangChain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
# Question - run the chain
print(rag_chain.invoke("What are the advantages of using RAG?"))

The advantages of using RAG (Retrieval-Augmented Generation) include:

1. **Improved Accuracy and Relevance**: RAG enhances the accuracy and relevance of responses by incorporating specific information from a database or dataset in real time, ensuring outputs are based on both the model’s pre-existing knowledge and the most current data.

2. **Customization and Flexibility**: RAG allows for tailored responses based on domain-specific needs by integrating a company's internal databases, creating personalized experiences and detailed outputs.

3. **Expanding Knowledge**: RAG helps expand the model’s knowledge beyond its training data, enabling organizations to fully utilize large language models (LLMs) by connecting them to relevant data.

4. **Product Recommendations**: It can enhance e-commerce product recommendation engines and generate personalized descriptions based on customer behavior.

5. **Training and Education**: RAG can customize learning materials for educational organizatio